# 3. Modeling & Evaluation

Este notebook entrena y compara XGBoost (sobre TF-IDF) y LSTM (sobre secuencias tokenizadas). Al final guarda el mejor modelo para producción.

In [14]:
import os, sys, subprocess
from getpass import getpass
from urllib.parse import quote

REPO_URL, REPO_DIR = "https://github.com/williamsnunezp/review-nlp-pipeline.git", "/content/review-nlp-pipeline"
token = os.getenv("GITHUB_TOKEN") or getpass("GitHub token (vacío si es público): ").strip()
url = REPO_URL.replace("https://", f"https://{quote(token, safe='')}@") if token else REPO_URL

if os.path.exists(REPO_DIR):
    print("No se clonó el repo: ya existe", REPO_DIR)
else:
    result = subprocess.run(["git", "clone", url, REPO_DIR])
    if result.returncode:
        print("No se clonó el repo.")
    else:
        print(os.listdir(REPO_DIR))
        print("Preparando entorno...")
        os.makedirs("/etc/OpenCL/vendors", exist_ok=True)
        with open("/etc/OpenCL/vendors/nvidia.icd", "w") as f:
            f.write("libnvidia-opencl.so.1")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "optuna", "optuna_integration"
        ])
        print("Entorno preparado.")

No se clonó el repo: ya existe /content/review-nlp-pipeline


In [15]:
import os
from getpass import getpass
from urllib.parse import quote
import pickle
import sys
import time
import warnings

import numpy as np
import optuna
from optuna_integration import XGBoostPruningCallback
# from optuna_integration import LightGBMPruningCallback
import pandas as pd
import tensorflow as tf
import xgboost as xgb
# import lightgbm as lgbm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, roc_auc_score, log_loss

warnings.filterwarnings("ignore")

In [7]:
# Configuración estricta de rutas para Google Colab
PROJECT_ROOT = '/content/review-nlp-pipeline'
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

## 1. Carga de Datos y Recursos
Cargaremos los archivos CSV limpios y cargaremos los archivos de serialización creados en el Notebook 2.

In [8]:
df_train = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/processed/train.csv')).dropna()
df_val = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/processed/val.csv')).dropna()
df_test = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/processed/test.csv')).dropna()

# Cargar vectorizador TF-IDF
with open(os.path.join(PROJECT_ROOT, 'models/vectorizer.pkl'), 'rb') as f:
    vectorizer = pickle.load(f)

# Cargar vocabulario LSTM
with open(os.path.join(PROJECT_ROOT, 'models/vocab.pkl'), 'rb') as f:
    word_to_idx = pickle.load(f)

print(f"Carga exitosa. Datos de entrenamiento: {df_train.shape[0]}, Vocabulario: {len(word_to_idx)}")

Carga exitosa. Datos de entrenamiento: 102000, Vocabulario: 11000


## 2. Preparación de Matrices TF-IDF (para Boosting)
Aplicaremos la transformación TF-IDF.

In [9]:
filas_test = []

X_train_tfidf = vectorizer.transform(df_train['cleaned_boosting'])
X_val_tfidf = vectorizer.transform(df_val['cleaned_boosting'])
X_test_tfidf = vectorizer.transform(df_test['cleaned_boosting'])

y_train = df_train['label'].values
y_val = df_val['label'].values
y_test = df_test['label'].values

print(f"Dimensiones TF-IDF de entrenamiento: {X_train_tfidf.shape}")

Dimensiones TF-IDF de entrenamiento: (102000, 5000)


## 3. XGBoost (Boosting)
Optimizamos los hiperparámetros con Optuna usando los conjuntos de entrenamiento y validación; luego evaluamos el modelo en test.

In [10]:
semilla = 42
N_TRIALS = 12
modelos_nombres = ["XGB"]

params = {
    "XGB": {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "device": "cuda",
        "max_bin": 64,
    }
}

Data_train = {
    "df_xgb_tr": xgb.QuantileDMatrix(X_train_tfidf, label=y_train, max_bin=64),
}
Data_val_test = {
    "df_xgb_val": xgb.QuantileDMatrix(X_val_tfidf, label=y_val, ref=Data_train["df_xgb_tr"], max_bin=64),
    "df_xgb_test": xgb.QuantileDMatrix(X_test_tfidf, label=y_test, ref=Data_train["df_xgb_tr"], max_bin=64),
}

# LightGBM desactivado:
# modelos_nombres = ["XGB", "LGBM"]
# params["LGBM"] = {
#     "objective": "binary", "device": "gpu", "metric": "auc",
#     "gpu_platform_id": 0, "gpu_device_id": 0, "min_child_weight": 1e-3,
#     "max_bin": 64, "verbosity": -1,
# }
# Data_train["df_lgb_tr"] = lgbm.Dataset(X_train_tfidf, label=y_train, free_raw_data=False)
# Data_val_test["df_lgb_val"] = lgbm.Dataset(X_val_tfidf, label=y_val, reference=Data_train["df_lgb_tr"], free_raw_data=False)
# Data_val_test["df_lgb_val_inf"] = X_val_tfidf
# Data_val_test["df_lgb_test_inf"] = X_test_tfidf


In [11]:
def get_param_space(nombre_modelo, trial):
    return {
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
    }


def predict_model(_modelo, Data_train, Data_val_test, params_nativos, semilla, num_boost_round, trial, calcular_val=True):
    train_params = {**params[_modelo], **params_nativos, "random_state": semilla}
    if calcular_val:
        callback = XGBoostPruningCallback(trial, "valid-auc")
        model = xgb.train(
            train_params, Data_train["df_xgb_tr"], num_boost_round=num_boost_round,
            evals=[(Data_val_test["df_xgb_val"], "valid")], callbacks=[callback], verbose_eval=False
        )
        y_prob = model.predict(Data_val_test["df_xgb_val"])
    else:
        model = xgb.train(train_params, Data_train["df_xgb_tr"], num_boost_round=num_boost_round, verbose_eval=False)
        y_prob = model.predict(Data_val_test["df_xgb_test"])
    return y_prob, (y_prob >= 0.5).astype(int), model

# Rama LightGBM desactivada:
# elif _modelo == "LGBM":
#     if calcular_val:
#         pruning_callback = LightGBMPruningCallback(trial, "auc", valid_name="valid_auc")
#         model = lgbm.train(
#             train_params, Data_train["df_lgb_tr"], num_boost_round=num_boost_round,
#             valid_sets=[Data_val_test["df_lgb_val"]], valid_names=["valid_auc"],
#             callbacks=[pruning_callback]
#         )
#         y_prob = model.predict(Data_val_test["df_lgb_val_inf"])
#     else:
#         model = lgbm.train(train_params, Data_train["df_lgb_tr"], num_boost_round=num_boost_round)
#         y_prob = model.predict(Data_val_test["df_lgb_test_inf"])
#     y_pred = (y_prob >= 0.5).astype(int)


def calculate_all_metrics(y_true, y_pred, y_proba=None):
    accuracy = accuracy_score(y_true, y_pred)
    precision_1, recall_1, f1_1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )
    precision_0, recall_0, f1_0, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=0, zero_division=0
    )
    roc_auc = roc_auc_score(y_true, y_proba) if y_proba is not None else None
    return {
        "Accuracy": accuracy, "Precision (1)": precision_1, "Recall (1)": recall_1,
        "F1 (1)": f1_1, "Precision (0)": precision_0, "Recall (0)": recall_0,
        "F1 (0)": f1_0, "ROC-AUC": roc_auc,
    }


In [12]:
best_hyperparams = {}
best_val_accuracy = {}
best_n_rounds = {}
trained_models = {}
boost_test_metrics = {}

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

t_inicio_global = time.time()

for nombre_modelo in modelos_nombres: 
    print(f"\n{'='*60}")
    print(f"  Optuna: {nombre_modelo} ({N_TRIALS} trials, AUC en validación)")
    print(f"{'='*60}")

    t_inicio = time.time()

    def make_objective(_modelo=nombre_modelo):
        def objective(trial):
            params_nativos = get_param_space(_modelo, trial)
            num_boost_round = trial.suggest_int('num_boost_round', 400, 1200)
            y_prob, y_pred, _ = predict_model(
                _modelo, Data_train, Data_val_test, params_nativos, semilla, num_boost_round, trial, calcular_val=True
            )
            return accuracy_score(y_val, y_pred)
        return objective

    study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=5, interval_steps=1))
    study.optimize(make_objective(), n_trials=N_TRIALS, show_progress_bar=True)

    best_params = study.best_params.copy()
    best_n_rounds[nombre_modelo] = best_params.pop('num_boost_round')
    best_hyperparams[nombre_modelo] = best_params
    best_val_accuracy[nombre_modelo] = study.best_value

    t_fin = time.time()
    print(f"\nMejor Accuracy val: {study.best_value:.4f}")
    print(f"Hiperparámetros: {best_params}")
    print(f"num_boost_round: {best_n_rounds[nombre_modelo]}")
    print(f"Tiempo: {t_fin - t_inicio:.1f}s")

print(f"\nTiempo total: {time.time() - t_inicio_global:.1f}s")

In [15]:
filas_test = []

for nombre_modelo in modelos_nombres:
    model_hyperparams = best_hyperparams[nombre_modelo]
    y_prob, y_pred, model = predict_model(
        nombre_modelo, Data_train, Data_val_test,
        model_hyperparams, semilla, best_n_rounds[nombre_modelo], trial=None, calcular_val=False
    )
    trained_models[nombre_modelo] = model
    metrics = calculate_all_metrics(y_test, y_pred, y_prob)
    boost_test_metrics[nombre_modelo] = metrics
    filas_test.append({'Modelo': nombre_modelo, **metrics})

df_test_results = pd.DataFrame(filas_test)
print("=" * 60)
print("  RESULTADOS EN TEST (Boosting)")
print("=" * 60)
display(df_test_results.style.highlight_max(
    subset=['Accuracy', 'F1 (1)', 'F1 (0)', 'ROC-AUC'],
    color='lightcoral'
))

## 4. Modelo 3: TensorFlow LSTM (Recurrent Neural Network)
Para el modelo recurrente, transformaremos las reseñas en secuencias numéricas de índices de vocabulario de longitud fija (padding).

In [16]:
def text_to_sequence_arr(texts, word_to_idx, max_len=100):
    sequences = []
    for text in texts:
        tokens = str(text).split()
        seq = [word_to_idx.get(tok, 1) for tok in tokens]
        seq = seq[:max_len] + [0] * max(0, max_len - len(seq))
        sequences.append(seq)
    # tf.int32 ocupa la mitad de memoria que el int64 por defecto de NumPy
    return tf.constant(sequences, dtype=tf.int32)

# Convertir textos a tf.Tensor de enteros (int32)
X_train_seq = text_to_sequence_arr(df_train['cleaned_rn'], word_to_idx)
X_val_seq   = text_to_sequence_arr(df_val['cleaned_rn'],   word_to_idx)
X_test_seq  = text_to_sequence_arr(df_test['cleaned_rn'],  word_to_idx)

In [18]:
from src.models import build_lstm

# Inicializacion y entrenamiento LSTM
BATCH_SIZE_LSTM = 128  
lstm_model = build_lstm(vocab_size=len(word_to_idx))
lstm_model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['AUC'])

# EarlyStopping
early_stop_lstm = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True, mode='min', verbose=1
)

lstm_model.fit(
    X_train_seq, y_train.astype("float32"),
    epochs=15,
    batch_size=BATCH_SIZE_LSTM,
    validation_data=(X_val_seq, y_val.astype("float32")),
    callbacks=[early_stop_lstm],
    verbose=1,
)

# Evaluacion LSTM
lstm_probs = lstm_model.predict(X_test_seq, batch_size=BATCH_SIZE_LSTM, verbose=0).flatten()
lstm_preds = (lstm_probs > 0.5).astype(int)
lstm_metrics = calculate_all_metrics(y_test, lstm_preds, lstm_probs)
filas_test = [row for row in filas_test if row.get('Modelo') != 'LSTM']
filas_test.append({'Modelo': 'LSTM', **lstm_metrics})
df_test_results = pd.DataFrame(filas_test)

print("=" * 60)
print("  RESULTADOS EN TEST (LSTM)")
print("=" * 60)
display(pd.DataFrame([filas_test[-1]]).style.highlight_max(
    subset=['Accuracy', 'F1 (1)', 'F1 (0)', 'ROC-AUC'],
    color='lightcoral'
))


Epoch 1/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - AUC: 0.8719 - loss: 0.4442 - val_AUC: 0.9489 - val_loss: 0.2948
Epoch 2/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - AUC: 0.9529 - loss: 0.2749 - val_AUC: 0.9592 - val_loss: 0.2722
Epoch 3/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - AUC: 0.9679 - loss: 0.2263 - val_AUC: 0.9635 - val_loss: 0.2942
Epoch 4/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - AUC: 0.9754 - loss: 0.1970 - val_AUC: 0.9675 - val_loss: 0.2631
Epoch 5/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - AUC: 0.9796 - loss: 0.1779 - val_AUC: 0.9686 - val_loss: 0.2351
Epoch 6/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - AUC: 0.9830 - loss: 0.1604 - val_AUC: 0.9691 - val_loss: 0.2293
Epoch 7/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - AUC: 0.9856 - loss: 0.1462 - val_AUC: 0.9691 - val_loss: 0.2410
Epoch 8/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - AUC: 0.9881 - loss: 0.1307 - val_AUC: 0.9686 - val_loss: 0.2616
Epoch 9/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 

,Modelo,Accuracy,Precision (1),Recall (1),F1 (1),Precision (0),Recall (0),F1 (0),ROC-AUC
0,LSTM,0.914292,0.912585,0.921253,0.916899,0.916126,0.906951,0.911515,0.968383


In [19]:
print("=" * 60)
print("  RESULTADO GENERAL EN TEST")
print("=" * 60)
display(df_test_results.style.highlight_max(
    subset=['Accuracy', 'F1 (1)', 'F1 (0)', 'ROC-AUC'],
    color='lightcoral'
))

In [ ]:
metric_cols = ["Accuracy", "F1 (1)", "F1 (0)", "ROC-AUC"]
wins = {}
for col in metric_cols:
    best_model = df_test_results.loc[df_test_results[col].idxmax(), "Modelo"]
    wins[best_model] = wins.get(best_model, 0) + 1

max_wins = max(wins.values())
candidatos = [name for name, count in wins.items() if count == max_wins]
best_model_name = (
    df_test_results[df_test_results["Modelo"].isin(candidatos)]
    .set_index("Modelo")["ROC-AUC"].idxmax()
)
print(f"\nMejor modelo: {best_model_name} ({max_wins}/{len(metric_cols)} métricas)")

models_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
if best_model_name == "LSTM":
    model_save_path = os.path.join(models_dir, "best_model.keras")
    lstm_model.save(model_save_path)
    model_type = "tf_lstm"
    with open(os.path.join(models_dir, "model_type.pkl"), "wb") as f:
        pickle.dump(model_type, f)
    print(f"Guardado lstm_model en: {model_save_path}")
else:
    model_save_path = os.path.join(models_dir, "best_model.pkl")
    with open(model_save_path, "wb") as f:
        pickle.dump(trained_models[best_model_name], f)
    print(f"Guardado modelo XGBoost en: {model_save_path}")
